# NF4 · Monitorización, fiabilidad y estabilidad (RA4)
## El servidor de juego · *juego online*

**Tu misión:** vigilar el pipeline que da servicio a los jugadores conectados.
No vas a configurar Grafana a mano: el stack viene **pre-provisionado** (Docker).
Tu trabajo de ingeniería es **observabilidad como código**: instrumentar, definir
**SLO**, escribir **alertas** y un **dashboard** en ficheros, y demostrar que el
sistema **detecta un incidente** automáticamente.


### Criterios del RA4
4.1/4.2 recolectar y visualizar métricas · 4.3 generar alertas · 4.4/4.5 fiabilidad
· 4.6 estabilidad del servicio.

> Autoevaluable y **sin capturas**: la evidencia son las métricas Prometheus y los
> ficheros `alertas.yml` y `dashboard.json`.

### Antes de empezar · levanta el entorno (una sola vez)

NF4 no genera datos: los produce el propio pipeline. Necesitas **dos terminales**, y cada
comando arranca **desde la raíz del repositorio**:

```bash
# Terminal 1 — stack de observabilidad (Prometheus :9090 · Grafana :3000)
cd nf4/monitorizacion && docker compose up -d

# Terminal 2 — el pipeline instrumentado
cd nf4 && python pipeline/pipeline.py --modo incidente --puerto 8000
```

El *datasource* y el dashboard de Grafana se cargan solos.

> **Lo que se evalúa NO depende de Docker.** Todas las celdas de este cuaderno leen las métricas
> **en proceso** y validan **ficheros**: funcionan con el stack apagado. Prometheus y Grafana
> están para que **veas** lo que estás midiendo, y merece la pena. Pero si Docker se te resiste,
> **sigue adelante con la actividad** y pide ayuda en el foro en paralelo: no te bloquea la nota.

Este cuaderno **se sitúa solo** en `nf4/`, así que todas sus rutas son relativas a esa
carpeta. Si la primera celda de código falla con un error de fichero no encontrado, es
que te falta este paso.


In [ ]:
import os
if os.path.basename(os.getcwd()) == "actividad": os.chdir("..")  # ejecutar desde la carpeta del núcleo (donde está datos/)
import json, sys, yaml
from pathlib import Path
from prometheus_client.parser import text_string_to_metric_families
sys.path.insert(0, "pipeline")
from pipeline import procesar, metricas_texto

MON = Path("monitorizacion")
resultados = {}
print("Listo. Recuerda: Grafana en :3000, Prometheus en :9090.")

---
### Comprobación del stack (opcional, pero hazla)

Antes de fiarte de un panel, comprueba que tu observabilidad **está observando**. Es parte del
oficio del RA4: un dashboard bonito alimentado por un *scrape* caído es peor que no tener nada.


In [ ]:
# Estado del stack. NO es evaluable y NO detiene el cuaderno: solo te dice qué tienes en pie.
import urllib.request, json as _json
def _vivo(url, nombre):
    try:
        with urllib.request.urlopen(url, timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

prom = _vivo("http://localhost:9090/-/ready", "Prometheus")
graf = _vivo("http://localhost:3000/api/health", "Grafana")
print(f"Prometheus :9090  -> {'en pie' if prom else 'no responde'}")
print(f"Grafana    :3000  -> {'en pie' if graf else 'no responde'}")

if prom:
    try:  # ¿está recogiendo de verdad las métricas del pipeline?
        with urllib.request.urlopen(
            "http://localhost:9090/api/v1/query?query=eventos_procesados_total", timeout=3) as r:
            n = len(_json.load(r)["data"]["result"])
        print(f"scrape del pipeline -> {'OK, ' + str(n) + ' serie(s)' if n else 'SIN datos: ¿arrancaste pipeline.py en la otra terminal?'}")
    except Exception as e:
        print("scrape del pipeline -> no se pudo consultar:", e)

if not (prom and graf):
    print("\nℹ️  El stack no está completo. Puedes hacer TODA la actividad igualmente:\n"
          "    las celdas siguientes leen las métricas en proceso y validan ficheros.")

---
## Fase 1 · Recolección de métricas (RA4.1/4.2)
Ejecuta el pipeline (modo normal) y lista las métricas Prometheus que expone.

In [ ]:
def leer_metricas(texto):
    vals, bases = {}, set()
    for fam in text_string_to_metric_families(texto):
        bases.add(fam.name)
        for s in fam.samples:
            vals[s.name] = s.value
    return vals, bases

_, bases = leer_metricas(metricas_texto(procesar("normal")))
resultados["metricas_expuestas"] = sorted(bases)
resultados["metricas_expuestas"]

---
## Fase 2 · SLI, SLO y evaluación de la alerta (RA4.3/4.4/4.5)

1. Escribe el **SLO** del servicio con **estos valores exactos**: tasa de error máx. **5 %**,
   frescura máx. **60 s**, p95 de latencia máx. **0.5 s**.
2. Implementa `evaluar_modo`: calcula la **SLI** de tasa de error y decide si la **alerta**
   debe activarse (error_rate por encima del SLO **o** frescura por encima del SLO).

> **Por qué se te dan los valores.** Un SLO real lo negocia el equipo con negocio; aquí se
> fijan para que el resultado de toda la clase sea comparable y la corrección automática
> pueda verificarlo. **Usa exactamente estos tres.** Si crees que deberían ser otros, ese es
> justo el debate que se valora en el razonamiento de la Fase 6, no aquí.


In [ ]:
# Fase 2 · SLO y evaluación de la alerta
SLO = {"error_rate_max": ..., "frescura_max_s": ..., "latencia_p95_max_s": ...}
# TODO: escribe los tres objetivos con los valores EXACTOS del enunciado:
#       error_rate_max = 0.05 · frescura_max_s = 60 · latencia_p95_max_s = 0.5

def evaluar_modo(modo):
    vals, _ = leer_metricas(metricas_texto(procesar(modo)))
    eventos = vals.get("eventos_procesados_total", 0)
    errores = vals.get("errores_total", 0)
    frescura = vals.get("datos_frescura_segundos", 0)
    error_rate = ...   # TODO: tasa de error · pista: errores/eventos, redondeado a 4 decimales
    alerta = ...       # TODO: ¿se viola algún SLO? · pista: error_rate > SLO["error_rate_max"] o frescura > SLO["frescura_max_s"]
    return {"eventos": int(eventos), "errores": int(errores), "error_rate": error_rate,
            "frescura_s": int(frescura), "alerta_activa": bool(alerta)}

resultados["slo"] = SLO
resultados["normal"] = evaluar_modo("normal")
resultados["incidente"] = evaluar_modo("incidente")
print("normal:", resultados["normal"]); print("incidente:", resultados["incidente"])

---
## Fase 3 · Alertas como código (RA4.3)

Abre `monitorizacion/alertas.yml` y **completa las dos reglas marcadas `# TODO`**
(`DatoObsoleto` y `LatenciaAlta`), copiando la estructura de `TasaErroresAlta`, que ya está
resuelta. Necesitas llegar a **3 reglas**.

Para validar la sintaxis, `promtool` **vive dentro del contenedor de Prometheus**, no en tu
terminal. Desde `nf4/monitorizacion`:

```bash
docker compose exec prometheus promtool check rules /etc/prometheus/alertas.yml
```

Esta celda comprueba además la estructura del fichero.

> **Tienes un ejemplo resuelto en tu propio repositorio.**
> `demo/nf4_smartcity/observabilidad/alertas.yml` trae **4 reglas completas** del caso de la
> masterclass. Cópiale la **estructura** (`expr`, `for`, `labels`, `annotations`); las métricas
> y los umbrales son los tuyos.


In [ ]:
def validar_alertas(path):
    doc = yaml.safe_load(open(path, encoding="utf-8"))
    reglas = [r for grp in doc.get("groups", []) for r in grp.get("rules", [])]
    ok = all(r.get("alert") and r.get("expr") and r.get("annotations") for r in reglas)
    return {"yaml_valida": bool(ok and reglas), "n_reglas": len(reglas),
            "nombres_reglas": sorted(r["alert"] for r in reglas if r.get("alert"))}

resultados["alertas"] = validar_alertas(MON / "alertas.yml")
resultados["alertas"]

---
## Fase 4 · Dashboard como código (RA4.2)

Abre `monitorizacion/grafana/dashboards/torre_control.json`. Trae **2 paneles resueltos**
—*Throughput* y *Tasa de error*— y te faltan **2 por añadir** hasta llegar a 4.

> **Ojo:** un `.json` **no admite comentarios**, así que ahí no vas a encontrar marcas `TODO`
> como en el `alertas.yml`. Los dos paneles que faltan son estos:

| Panel a añadir | SLI | Expresión PromQL |
|---|---|---|
| **Latencia p95** | latencia | `histogram_quantile(0.95, rate(latencia_procesamiento_segundos_bucket[1m]))` |
| **Frescura del dato** | frescura | `datos_frescura_segundos` |

Copia la estructura de un panel existente y cámbiale `title`, `id` y el `expr` del `target`.
Grafana lo recarga solo. Esta celda valida la estructura.

> **Y aquí también.** `demo/nf4_smartcity/observabilidad/torre_control.json` tiene **4 paneles
> resueltos**. Ábrelo al lado del tuyo: verás cómo se anida `panels → targets → expr` sin tener
> que adivinarlo.


In [ ]:
def validar_dashboard(path):
    d = json.load(open(path, encoding="utf-8"))
    paneles = d.get("panels", [])
    exprs = " ".join(t.get("expr","") for p in paneles for t in p.get("targets", []))
    refs = sorted({m for m in ["eventos_procesados_total","errores_total",
                   "latencia_procesamiento_segundos","datos_frescura_segundos"] if m in exprs})
    return {"valido": bool(d.get("title") and len(paneles) >= 4),
            "n_paneles": len(paneles), "metricas_referenciadas": refs}

resultados["dashboard"] = validar_dashboard(MON / "grafana/dashboards/torre_control.json")
resultados["dashboard"]

---
## Fase 5 · Análisis del incidente (RA4.6) — texto

Arranca el pipeline en modo `incidente` y observa la degradación. **Dos formas, la nota es la
misma:**

- **Con Grafana** (recomendado, `localhost:3000`): ves las cuatro series moverse a la vez, que
  es justo lo que un dashboard aporta sobre una tabla de números.
- **Sin Grafana**: te bastan los valores que ya calculaste en la Fase 2 para `normal` e
  `incidente` —tasa de error, frescura y latencia—. Son los mismos números que dibujaría el panel.

Escribe **5-8 líneas**:

- ¿Qué SLO se viola **primero** y por qué? *(compara cuánto se aleja cada métrica de su umbral,
  no su valor absoluto)*
- ¿Qué alerta se activa y con qué severidad?
- ¿Qué medida tomarías para **restaurar la estabilidad** del servicio, y a cambio de qué?

*(Escribe aquí tu análisis.)*


---
## Fase 6 · Razonamiento (formato examen, RA4)
En **máximo 12 líneas**, con terminología precisa:

**a)** Diferencia un **log** de una **métrica** e indica cuándo es preferible cada uno.

**b)** Define **SLI** y **SLO** con un ejemplo concreto para este pipeline en tiempo real.

**c)** Explica la función de **Prometheus** como recolector (*scraping*) y razona por
qué las **alertas automáticas** son imprescindibles para una red de jugadores crítica.

*(Escribe aquí tu respuesta.)*


---
## Celda final · Generar `resultados.json` (no modificar)

In [ ]:
ALUMNO = "TU_NOMBRE_Y_APELLIDOS"   # <-- pon aquí "Apellidos, Nombre"
resultados["metadata"] = {"caso":"gaming","seed":42,"alumno":ALUMNO}
assert ALUMNO != "TU_NOMBRE_Y_APELLIDOS", "⚠️ Pon tus Apellidos, Nombre en ALUMNO antes de entregar."
assert set(resultados) >= {"metricas_expuestas","slo","normal","incidente","alertas","dashboard"}, "Faltan secciones"
json.dump(resultados, open("resultados.json","w",encoding="utf-8"), ensure_ascii=False, indent=2)
print("✅ resultados.json generado. Entrega el .ipynb, alertas.yml y torre_control.json.")

---
### Checkpoint de preparación al examen (no evaluable)
1. ¿Por qué una métrica es mejor que un log para alertar sobre la tasa de error?
2. Tu SLO de frescura es 60s y el dato lleva 5 min sin actualizarse: ¿qué pasa?
3. ¿Qué hace exactamente Prometheus cada 5 segundos contra tu pipeline?